# Neural Network Fusion Architectures: Combining Outputs from Multiple Networks

## Introduction

In modern deep learning, many tasks require integrating information from **multiple sources, modalities, or representation spaces**. Multi-stream architectures — networks with two or more parallel branches — are ubiquitous in computer vision, natural language processing, multimodal learning, and beyond.

The central design question in such architectures is: **how do we combine the outputs of these parallel streams into a unified representation?**

This notebook provides a comprehensive treatment of fusion strategies, organized from simplest to most expressive:

| # | Technique | Key Idea | Complexity |
|---|-----------|----------|------------|
| 1 | Late Fusion (Concatenation) | Stack feature vectors | $$O(d_1 + d_2)$$ |
| 2 | Element-wise Operations | Add, multiply, or average | $$O(d)$$ |
| 3 | Bilinear Fusion | Model pairwise feature interactions | $$O(d_1 \cdot d_2)$$ |
| 4 | Gated Fusion | Learn dynamic, input-dependent weighting | $$O(d)$$ |
| 5 | Attention-based Fusion | Soft alignment and weighting | $$O(d)$$ |
| 6 | Tensor Fusion | Outer product of augmented vectors | $$O(d_1 \cdot d_2)$$ |
| 7 | Cross-Attention | Transformer-style cross-stream attention | $$O(n^2 \cdot d)$$ |
| 8 | Feature-wise Linear Modulation (FiLM) | Conditional affine transforms | $$O(d)$$ |
| 9 | Mixture of Experts (MoE) Fusion | Route inputs to specialized sub-networks | $$O(k \cdot d)$$ |

---

## Notation

Throughout this notebook:

- $$\mathbf{h}_1 \in \mathbb{R}^{d_1}$$ and $$\mathbf{h}_2 \in \mathbb{R}^{d_2}$$ — output vectors from two sub-networks (branches)
- $$\mathbf{W}$$, $$\mathbf{b}$$ — learnable weight matrices and bias vectors
- $$\sigma(\cdot)$$ — sigmoid activation function (unless stated otherwise)
- $$\oplus$$ — concatenation operator
- $$\odot$$ — element-wise (Hadamard) product
- $$\otimes$$ — outer product

In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)

# ---------------------------------------------------------------------------
# Utility: Two dummy sub-networks (branches) that simulate feature extractors
# ---------------------------------------------------------------------------

class BranchA(nn.Module):
    """Simulates a sub-network producing a d-dimensional embedding.
    Think: image encoder, text encoder, audio feature extractor, etc."""
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )
    def forward(self, x):
        return self.net(x)


class BranchB(nn.Module):
    """Simulates a second sub-network producing a d-dimensional embedding."""
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )
    def forward(self, x):
        return self.net(x)


# Instantiate branches
branch_a = BranchA(input_dim=50, output_dim=64)
branch_b = BranchB(input_dim=30, output_dim=64)

# Create dummy inputs (batch_size=8)
x_a = torch.randn(8, 50)  # e.g., image features
x_b = torch.randn(8, 30)  # e.g., text features

# Get branch outputs
h1 = branch_a(x_a)
h2 = branch_b(x_b)

print(f"Branch A output shape: {h1.shape}")  # [8, 64]
print(f"Branch B output shape: {h2.shape}")  # [8, 64]
print(f"\nThese h1, h2 tensors will be reused in all fusion examples below.")

# 1. Late Fusion (Concatenation)

## Concept

**Late Fusion** is the simplest and most widely used fusion strategy. Each sub-network processes its input independently up to a certain depth, and only at the end are the representations combined — typically via **concatenation** followed by one or more fully-connected layers.

## Mathematical Formulation

Given branch outputs $$\mathbf{h}_1 \in \mathbb{R}^{d_1}$$ and $$\mathbf{h}_2 \in \mathbb{R}^{d_2}$$:

$$\mathbf{h}_{\text{fused}} = [\mathbf{h}_1 \oplus \mathbf{h}_2] = [\mathbf{h}_1; \mathbf{h}_2] \in \mathbb{R}^{d_1 + d_2}$$

The fused vector is then projected to the desired output dimension:

$$\mathbf{y} = f(\mathbf{W} \cdot \mathbf{h}_{\text{fused}} + \mathbf{b})$$

where $$\mathbf{W} \in \mathbb{R}^{d_{\text{out}} \times (d_1 + d_2)}$$ and $$f$$ is an activation function.

## Properties

| Property | Description |
|----------|-------------|
| **Expressiveness** | The downstream MLP can learn arbitrary interactions between features from both branches |
| **Parameter cost** | $$O(d_{\text{out}} \times (d_1 + d_2))$$ in the first fusion layer |
| **When to use** | Default choice; works well when branches produce semantically different features |
| **Limitation** | Does not explicitly model interactions — relies entirely on subsequent layers to discover them |

## Industrial Applications

| Industry | Application | Details |
|----------|-------------|---------|
| **Autonomous Driving** | Sensor fusion (LiDAR + Camera) | Waymo, Tesla combine 3D point-cloud features with 2D image CNN features via concatenation before detection heads |
| **E-commerce / Retail** | Product search ranking | Concatenate query text embeddings with product image embeddings for relevance scoring (e.g., Amazon, Alibaba) |
| **Healthcare** | Medical diagnosis | Fuse patient EHR tabular features with radiology image features for disease classification |
| **Social Media** | Content moderation | Combine text (BERT) + image (ResNet) features to detect harmful multimodal content (Facebook/Meta) |
| **Robotics** | Grasp prediction | Concatenate tactile sensor features with visual features for stable grasp planning |

In [0]:
class LateFusionModel(nn.Module):
    """
    Late Fusion: Concatenate branch outputs and pass through an MLP.
    
    Architecture:
        h1 (d1) --\
                   --> [h1; h2] (d1+d2) --> MLP --> output
        h2 (d2) --/
    """
    def __init__(self, d1: int, d2: int, hidden_dim: int, output_dim: int):
        super().__init__()
        self.fusion_mlp = nn.Sequential(
            nn.Linear(d1 + d2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, h1: torch.Tensor, h2: torch.Tensor) -> torch.Tensor:
        # Concatenate along feature dimension
        h_fused = torch.cat([h1, h2], dim=-1)  # [B, d1+d2]
        return self.fusion_mlp(h_fused)


# Demonstration
late_fusion = LateFusionModel(d1=64, d2=64, hidden_dim=128, output_dim=10)
output = late_fusion(h1, h2)

print(f"h1 shape:      {h1.shape}")
print(f"h2 shape:      {h2.shape}")
print(f"Fused shape:   {torch.cat([h1, h2], dim=-1).shape}")
print(f"Output shape:  {output.shape}")
print(f"\nParameters:    {sum(p.numel() for p in late_fusion.parameters()):,}")

# 2. Element-wise Operations (Addition, Multiplication, Averaging)

## Concept

When both branches produce vectors of the **same dimensionality** ($$d_1 = d_2 = d$$), we can combine them via simple element-wise operations. This is parameter-free at the fusion point and preserves the original dimensionality.

## Mathematical Formulations

### 2.1 Additive Fusion

$$\mathbf{h}_{\text{fused}} = \mathbf{h}_1 + \mathbf{h}_2 \in \mathbb{R}^d$$

Interpretation: Features from both streams contribute **additively** to the combined representation. Widely used in ResNets (skip connections are a form of additive fusion).

### 2.2 Multiplicative (Hadamard) Fusion

$$\mathbf{h}_{\text{fused}} = \mathbf{h}_1 \odot \mathbf{h}_2 \in \mathbb{R}^d$$

Interpretation: Each dimension is **gated** by the corresponding dimension of the other stream. This captures **pairwise interactions** between aligned dimensions. If one stream is near zero in some dimension, it suppresses that dimension in the output.

### 2.3 Average Fusion

$$\mathbf{h}_{\text{fused}} = \frac{\mathbf{h}_1 + \mathbf{h}_2}{2} \in \mathbb{R}^d$$

### 2.4 Maximum Fusion

$$\mathbf{h}_{\text{fused}}^{(i)} = \max(\mathbf{h}_1^{(i)},\; \mathbf{h}_2^{(i)}) \quad \forall i \in \{1, \ldots, d\}$$

## Comparison

| Operation | Preserves magnitude? | Captures interactions? | Parameters |
|-----------|---------------------|----------------------|------------|
| Addition | Yes (can grow) | No (linear) | 0 |
| Multiplication | Depends on values | Yes (second-order) | 0 |
| Average | Yes (bounded) | No (linear) | 0 |
| Maximum | Yes | No (selection) | 0 |

## Industrial Applications

| Industry | Application | Fusion Type | Details |
|----------|-------------|-------------|---------|
| **Computer Vision** | ResNets / Skip Connections | Addition | Residual connections ($$\mathbf{h} + F(\mathbf{h})$$) used universally in production CNNs at Google, Meta, NVIDIA |
| **NLP / Search** | Siamese Networks for similarity | Multiplication | Sentence-BERT uses Hadamard product of sentence embeddings for semantic similarity at scale (Elastic, Pinecone) |
| **Recommendation Systems** | Neural Collaborative Filtering | Multiplication | Element-wise product of user and item embeddings captures interaction signals (Netflix, Spotify) |
| **Speech Recognition** | Feature combination | Addition | Adding acoustic features from parallel CNN and RNN streams in hybrid ASR systems (Google Assistant, Siri) |
| **Fraud Detection** | Transaction scoring | Maximum | Max-pooling over multiple behavioral feature extractors to capture worst-case anomaly signals (PayPal, Stripe) |

In [0]:
class ElementWiseFusion(nn.Module):
    """
    Element-wise fusion with optional learned scaling.
    Requires d1 == d2. If dimensions differ, projects to a common space first.
    """
    def __init__(self, d1: int, d2: int, mode: str = "add", output_dim: int = 10):
        super().__init__()
        assert mode in ["add", "multiply", "average", "maximum"], f"Unknown mode: {mode}"
        self.mode = mode
        
        # Project to common dimension if needed
        self.d_common = max(d1, d2)
        self.proj1 = nn.Linear(d1, self.d_common) if d1 != self.d_common else nn.Identity()
        self.proj2 = nn.Linear(d2, self.d_common) if d2 != self.d_common else nn.Identity()
        
        # Output head
        self.output_head = nn.Linear(self.d_common, output_dim)
    
    def forward(self, h1: torch.Tensor, h2: torch.Tensor) -> torch.Tensor:
        h1 = self.proj1(h1)
        h2 = self.proj2(h2)
        
        if self.mode == "add":
            h_fused = h1 + h2
        elif self.mode == "multiply":
            h_fused = h1 * h2
        elif self.mode == "average":
            h_fused = (h1 + h2) / 2.0
        elif self.mode == "maximum":
            h_fused = torch.max(h1, h2)
        
        return self.output_head(h_fused)


# Demonstrate all modes
print("Element-wise Fusion Outputs:")
print("-" * 50)
for mode in ["add", "multiply", "average", "maximum"]:
    model = ElementWiseFusion(d1=64, d2=64, mode=mode, output_dim=10)
    out = model(h1, h2)
    
    # Show intermediate fused representation statistics
    with torch.no_grad():
        if mode == "add":
            fused = h1 + h2
        elif mode == "multiply":
            fused = h1 * h2
        elif mode == "average":
            fused = (h1 + h2) / 2.0
        elif mode == "maximum":
            fused = torch.max(h1, h2)
    
    print(f"  {mode:12s} | fused mean: {fused.mean():.4f}, std: {fused.std():.4f} | output: {out.shape}")

# 3. Bilinear Fusion

## Concept

**Bilinear Fusion** explicitly models **second-order interactions** between every pair of features from the two streams. Unlike element-wise multiplication (which only captures interactions between aligned dimensions), bilinear fusion captures **all cross-stream pairwise interactions**.

## Mathematical Formulation

### Full Bilinear Pooling

For each output dimension $$k$$:

$$y_k = \mathbf{h}_1^\top \mathbf{W}_k \mathbf{h}_2 + b_k$$

where $$\mathbf{W}_k \in \mathbb{R}^{d_1 \times d_2}$$ is a learnable matrix for the $$k$$-th output.

In vectorized form for all $$K$$ outputs:

$$\mathbf{y} = \text{Bilinear}(\mathbf{h}_1, \mathbf{h}_2) = \sum_{i,j} \mathbf{W}_{:,i,j} \cdot h_1^{(i)} \cdot h_2^{(j)} + \mathbf{b}$$

where $$\mathbf{W} \in \mathbb{R}^{K \times d_1 \times d_2}$$ is a 3D tensor.

### Low-Rank Bilinear Fusion

The full bilinear form has $$O(K \cdot d_1 \cdot d_2)$$ parameters — prohibitive for large dimensions. The **low-rank factorization** approximates:

$$\mathbf{W}_k \approx \mathbf{U}_k \mathbf{V}_k^\top$$

where $$\mathbf{U}_k \in \mathbb{R}^{d_1 \times r}$$ and $$\mathbf{V}_k \in \mathbb{R}^{d_2 \times r}$$, with rank $$r \ll \min(d_1, d_2)$$.

This simplifies computation to:

$$y_k = \mathbf{1}^\top (\mathbf{U}_k^\top \mathbf{h}_1 \odot \mathbf{V}_k^\top \mathbf{h}_2)$$

## Properties

| Variant | Parameters | Expressiveness | Use case |
|---------|-----------|----------------|----------|
| Full bilinear | $$O(K \cdot d_1 \cdot d_2)$$ | Very high | Small dims |
| Low-rank bilinear | $$O(K \cdot r \cdot (d_1 + d_2))$$ | High | Large dims |
| Factorized bilinear | $$O(r \cdot (d_1 + d_2))$$ | Moderate | Efficiency |

## Industrial Applications

| Industry | Application | Details |
|----------|-------------|---------|
| **Visual Question Answering (VQA)** | Image-Question interaction | MCB/MLB pooling used in production VQA systems to model fine-grained interactions between image regions and question words (Google Lens, visual search) |
| **Fine-grained Recognition** | Product classification | Bilinear CNN models distinguish subtle visual differences (e.g., bird species, car models) for e-commerce cataloging (eBay, Wayfair) |
| **Remote Sensing** | Land-use classification | Bilinear pooling captures texture-shape interactions in satellite imagery for agricultural monitoring and urban planning |
| **Drug Discovery** | Protein-ligand interaction | Bilinear models capture pairwise interactions between protein residue features and molecular fingerprints for binding affinity prediction (DeepMind, Schrödinger) |
| **Ad Tech** | Click-through rate prediction | Low-rank bilinear layers model user-feature × ad-feature interactions in real-time bidding (Google Ads, Meta Ads) |

In [0]:
class BilinearFusion(nn.Module):
    """
    Full Bilinear Fusion using PyTorch's nn.Bilinear.
    
    Computes: y_k = h1^T W_k h2 + b_k for each output dimension k.
    """
    def __init__(self, d1: int, d2: int, output_dim: int):
        super().__init__()
        self.bilinear = nn.Bilinear(d1, d2, output_dim)
    
    def forward(self, h1: torch.Tensor, h2: torch.Tensor) -> torch.Tensor:
        return self.bilinear(h1, h2)


class LowRankBilinearFusion(nn.Module):
    """
    Low-Rank Bilinear Fusion.
    
    Factorizes W_k = U_k V_k^T to reduce parameters from O(K*d1*d2) to O(K*r*(d1+d2)).
    Computation: y = sum_pool(U^T h1 ⊙ V^T h2) followed by output projection.
    """
    def __init__(self, d1: int, d2: int, output_dim: int, rank: int = 16):
        super().__init__()
        self.rank = rank
        
        # Low-rank projections
        self.U = nn.Linear(d1, rank, bias=False)  # Project h1 to rank-dim space
        self.V = nn.Linear(d2, rank, bias=False)  # Project h2 to rank-dim space
        
        # Output projection
        self.output = nn.Linear(rank, output_dim)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, h1: torch.Tensor, h2: torch.Tensor) -> torch.Tensor:
        # Project both inputs to rank-dimensional space
        h1_proj = self.U(h1)          # [B, rank]
        h2_proj = self.V(h2)          # [B, rank]
        
        # Hadamard product captures bilinear interactions in the low-rank space
        fused = h1_proj * h2_proj     # [B, rank]
        fused = self.dropout(F.relu(fused))
        
        return self.output(fused)     # [B, output_dim]


# Compare full vs low-rank bilinear
full_bilinear = BilinearFusion(d1=64, d2=64, output_dim=10)
low_rank_bilinear = LowRankBilinearFusion(d1=64, d2=64, output_dim=10, rank=16)

out_full = full_bilinear(h1, h2)
out_lr = low_rank_bilinear(h1, h2)

params_full = sum(p.numel() for p in full_bilinear.parameters())
params_lr = sum(p.numel() for p in low_rank_bilinear.parameters())

print(f"Full Bilinear:")
print(f"  Output shape: {out_full.shape}")
print(f"  Parameters:   {params_full:,}  (W is {64}×{64}×{10} = {64*64*10:,} + bias)")
print(f"\nLow-Rank Bilinear (rank=16):")
print(f"  Output shape: {out_lr.shape}")
print(f"  Parameters:   {params_lr:,}  ({params_full/params_lr:.1f}× fewer)")
print(f"\nParameter reduction: {(1 - params_lr/params_full)*100:.1f}%")

# 4. Gated Fusion

## Concept

**Gated Fusion** learns a dynamic, input-dependent weighting between the two streams. Rather than treating both branches equally (as in addition or averaging), the network learns **when to trust each branch** based on the current input.

This is particularly useful when the reliability or informativeness of each modality varies across samples.

## Mathematical Formulation

### Simple Gated Fusion

A scalar or vector gate $$\mathbf{g} \in [0, 1]^d$$ is computed from both inputs:

$$\mathbf{g} = \sigma(\mathbf{W}_g [\mathbf{h}_1 \oplus \mathbf{h}_2] + \mathbf{b}_g)$$

The fused representation is a convex combination:

$$\mathbf{h}_{\text{fused}} = \mathbf{g} \odot \mathbf{h}_1 + (1 - \mathbf{g}) \odot \mathbf{h}_2$$

where $$\sigma$$ is the sigmoid function ensuring $$\mathbf{g} \in [0,1]^d$$.

### Interpretation

- $$g^{(i)} \approx 1$$: The $$i$$-th dimension relies heavily on branch 1
- $$g^{(i)} \approx 0$$: The $$i$$-th dimension relies heavily on branch 2
- $$g^{(i)} \approx 0.5$$: Equal contribution from both branches

### Multi-Gate Variant

A more expressive variant uses separate gates for each branch and a residual:

$$\mathbf{g}_1 = \sigma(\mathbf{W}_1 [\mathbf{h}_1 \oplus \mathbf{h}_2] + \mathbf{b}_1)$$

$$\mathbf{g}_2 = \sigma(\mathbf{W}_2 [\mathbf{h}_1 \oplus \mathbf{h}_2] + \mathbf{b}_2)$$

$$\mathbf{h}_{\text{fused}} = \mathbf{g}_1 \odot \mathbf{h}_1 + \mathbf{g}_2 \odot \mathbf{h}_2$$

Here $$\mathbf{g}_1$$ and $$\mathbf{g}_2$$ are **independent** — they need not sum to 1, allowing both suppression and amplification.

## Connection to Other Architectures

- **LSTM**: The forget/input gates are a form of gated fusion between the previous state and new input
- **GRU**: The update gate interpolates between previous hidden state and candidate
- **Highway Networks**: Gate between transformed and original input

## Industrial Applications

| Industry | Application | Details |
|----------|-------------|---------|
| **Multimodal Sentiment Analysis** | Product reviews (text + images) | Gated fusion dynamically suppresses the noisy modality (e.g., irrelevant stock images) while emphasizing the informative one |
| **Autonomous Vehicles** | Sensor reliability weighting | Gates learn to downweight LiDAR in heavy rain or camera in darkness, fusing only reliable sensor streams (Mobileye, Cruise) |
| **Clinical NLP** | EHR + clinical notes | Gate suppresses structured data when notes contain richer context and vice versa — adapts per patient (Epic, Tempus) |
| **Video Understanding** | Audio-visual fusion | YouTube, TikTok content classifiers gate audio vs. visual streams depending on content type (music video vs. silent tutorial) |
| **Industrial IoT** | Predictive maintenance | Gated fusion of vibration sensor data and thermal imaging — gates learn which sensor is more predictive for each failure mode (Siemens, GE) |

In [0]:
class GatedFusion(nn.Module):
    """
    Gated Fusion: Learn input-dependent weights for combining two streams.
    
    g = σ(W_g [h1; h2] + b_g)
    h_fused = g ⊙ h1 + (1-g) ⊙ h2
    """
    def __init__(self, d1: int, d2: int, output_dim: int):
        super().__init__()
        d = max(d1, d2)
        self.proj1 = nn.Linear(d1, d) if d1 != d else nn.Identity()
        self.proj2 = nn.Linear(d2, d) if d2 != d else nn.Identity()
        
        # Gate network: takes concatenation, outputs gate values
        self.gate = nn.Sequential(
            nn.Linear(d * 2, d),
            nn.Sigmoid()
        )
        
        self.output_head = nn.Linear(d, output_dim)
    
    def forward(self, h1: torch.Tensor, h2: torch.Tensor) -> torch.Tensor:
        h1 = self.proj1(h1)  # [B, d]
        h2 = self.proj2(h2)  # [B, d]
        
        # Compute gate from both streams
        g = self.gate(torch.cat([h1, h2], dim=-1))  # [B, d], values in [0, 1]
        
        # Gated combination
        h_fused = g * h1 + (1 - g) * h2  # [B, d]
        
        return self.output_head(h_fused)
    
    def get_gate_values(self, h1: torch.Tensor, h2: torch.Tensor) -> torch.Tensor:
        """Utility to inspect gate values for interpretability."""
        h1 = self.proj1(h1)
        h2 = self.proj2(h2)
        return self.gate(torch.cat([h1, h2], dim=-1))


class MultiGateFusion(nn.Module):
    """
    Multi-Gate Fusion: Independent gates for each branch (no sum-to-1 constraint).
    
    g1 = σ(W1 [h1; h2] + b1)
    g2 = σ(W2 [h1; h2] + b2)
    h_fused = g1 ⊙ h1 + g2 ⊙ h2
    """
    def __init__(self, d1: int, d2: int, output_dim: int):
        super().__init__()
        d = max(d1, d2)
        self.proj1 = nn.Linear(d1, d) if d1 != d else nn.Identity()
        self.proj2 = nn.Linear(d2, d) if d2 != d else nn.Identity()
        
        self.gate1 = nn.Sequential(nn.Linear(d * 2, d), nn.Sigmoid())
        self.gate2 = nn.Sequential(nn.Linear(d * 2, d), nn.Sigmoid())
        
        self.output_head = nn.Linear(d, output_dim)
    
    def forward(self, h1: torch.Tensor, h2: torch.Tensor) -> torch.Tensor:
        h1 = self.proj1(h1)
        h2 = self.proj2(h2)
        
        combined = torch.cat([h1, h2], dim=-1)
        g1 = self.gate1(combined)
        g2 = self.gate2(combined)
        
        h_fused = g1 * h1 + g2 * h2
        return self.output_head(h_fused)


# Demonstration
gated = GatedFusion(d1=64, d2=64, output_dim=10)
multi_gated = MultiGateFusion(d1=64, d2=64, output_dim=10)

out_gated = gated(h1, h2)
out_multi = multi_gated(h1, h2)

# Inspect gate values
with torch.no_grad():
    gate_vals = gated.get_gate_values(h1, h2)

print(f"Gated Fusion output:       {out_gated.shape}")
print(f"Multi-Gate Fusion output:  {out_multi.shape}")
print(f"\nGate statistics (how much the model relies on branch 1):")
print(f"  Mean gate value: {gate_vals.mean():.4f}")
print(f"  Std gate value:  {gate_vals.std():.4f}")
print(f"  Min: {gate_vals.min():.4f}, Max: {gate_vals.max():.4f}")
print(f"  (0.5 = equal weight, >0.5 = prefer branch 1, <0.5 = prefer branch 2)")

# 5. Attention-based Fusion

## Concept

**Attention-based Fusion** extends gating by computing attention weights that determine the relative importance of each stream. While gating operates dimension-wise, attention typically computes a **scalar weight per stream** (or per token/region) and takes a weighted combination.

## Mathematical Formulation

### Stream-level Attention

Given $$N$$ branch outputs $$\{\mathbf{h}_1, \mathbf{h}_2, \ldots, \mathbf{h}_N\}$$, compute attention scores:

$$e_i = \mathbf{w}^\top \tanh(\mathbf{W}_a \mathbf{h}_i + \mathbf{b}_a) \quad \text{for } i = 1, \ldots, N$$

Normalize via softmax:

$$\alpha_i = \frac{\exp(e_i)}{\sum_{j=1}^N \exp(e_j)}$$

Compute the fused representation:

$$\mathbf{h}_{\text{fused}} = \sum_{i=1}^N \alpha_i \mathbf{h}_i$$

### Self-Attention over Streams (Squeeze-and-Excitation style)

Alternatively, use a shared projection to score each stream:

$$\mathbf{s} = \text{softmax}\left(\mathbf{W}_2 \cdot \text{ReLU}(\mathbf{W}_1 \cdot \bar{\mathbf{h}})\right)$$

where $$\bar{\mathbf{h}}$$ is the mean (or concatenation) of all streams, and $$\mathbf{s} \in \mathbb{R}^N$$ gives the per-stream weights.

## Difference from Gating

| Aspect | Gating | Attention |
|--------|--------|-----------|
| Granularity | Per-dimension weights | Per-stream (scalar) weights |
| Normalization | Independent sigmoids | Softmax (sums to 1) |
| Constraint | Complementary (g and 1-g) | Competitive (softmax) |
| Interpretation | "How much of each dim" | "Which stream overall" |

## Industrial Applications

| Industry | Application | Details |
|----------|-------------|---------|
| **Multimodal AI Assistants** | Multi-sensor fusion | Attention-weighted fusion of speech, gesture, and gaze streams in AR/VR assistants (Apple Vision Pro, Meta Quest) |
| **Financial Trading** | Multi-source signal fusion | Attention over news sentiment, order-book features, and technical indicators — learn which source is most informative per market regime (Two Sigma, Citadel) |
| **Genomics** | Multi-omics integration | Attention-weighted fusion of transcriptomics, proteomics, and metabolomics for disease subtyping (Illumina, 23andMe) |
| **Smart Manufacturing** | Quality inspection | SE-style attention over multiple camera angles and sensor modalities to focus on the most informative view per defect type (Bosch, Foxconn) |
| **Disaster Response** | Multi-modal satellite analysis | Attention over optical, SAR, and infrared satellite streams — model learns which modality is most informative given cloud cover, time of day (Planet Labs, Maxar) |

In [0]:
class AttentionFusion(nn.Module):
    """
    Attention-based Fusion over N streams.
    
    Computes a scalar attention weight per stream and takes a weighted sum.
    Generalizes to any number of branches.
    
    e_i = w^T tanh(W_a h_i + b_a)
    α_i = softmax(e_i)
    h_fused = Σ α_i * h_i
    """
    def __init__(self, d: int, num_streams: int = 2, output_dim: int = 10):
        super().__init__()
        self.d = d
        self.num_streams = num_streams
        
        # Attention scoring network
        self.attention_net = nn.Sequential(
            nn.Linear(d, d // 2),
            nn.Tanh(),
            nn.Linear(d // 2, 1)  # Scalar score per stream
        )
        
        self.output_head = nn.Linear(d, output_dim)
    
    def forward(self, *streams: torch.Tensor) -> torch.Tensor:
        """
        Args:
            *streams: Variable number of tensors, each [B, d]
        Returns:
            Output tensor [B, output_dim]
        """
        # Stack streams: [B, N, d]
        stacked = torch.stack(streams, dim=1)
        
        # Compute attention scores: [B, N, 1]
        scores = self.attention_net(stacked)
        
        # Softmax over streams dimension: [B, N, 1]
        alpha = F.softmax(scores, dim=1)
        
        # Weighted sum: [B, d]
        h_fused = (alpha * stacked).sum(dim=1)
        
        return self.output_head(h_fused), alpha.squeeze(-1)


class SqueezeExcitationFusion(nn.Module):
    """
    SE-style Attention Fusion.
    
    Uses global context (mean of streams) to compute per-stream importance.
    Inspired by Squeeze-and-Excitation Networks (Hu et al., 2018).
    """
    def __init__(self, d: int, num_streams: int = 2, reduction: int = 4, output_dim: int = 10):
        super().__init__()
        self.se = nn.Sequential(
            nn.Linear(d * num_streams, d * num_streams // reduction),
            nn.ReLU(),
            nn.Linear(d * num_streams // reduction, num_streams),
            nn.Softmax(dim=-1)
        )
        self.output_head = nn.Linear(d, output_dim)
    
    def forward(self, *streams: torch.Tensor) -> torch.Tensor:
        # Global context from all streams
        context = torch.cat(streams, dim=-1)  # [B, N*d]
        
        # Compute stream importance
        weights = self.se(context)  # [B, N]
        
        # Stack and weight
        stacked = torch.stack(streams, dim=1)  # [B, N, d]
        h_fused = (weights.unsqueeze(-1) * stacked).sum(dim=1)  # [B, d]
        
        return self.output_head(h_fused), weights


# Demonstration
attn_fusion = AttentionFusion(d=64, num_streams=2, output_dim=10)
se_fusion = SqueezeExcitationFusion(d=64, num_streams=2, output_dim=10)

out_attn, attn_weights = attn_fusion(h1, h2)
out_se, se_weights = se_fusion(h1, h2)

print("Attention Fusion:")
print(f"  Output shape: {out_attn.shape}")
print(f"  Attention weights (first 4 samples):")
print(f"    Branch 1: {attn_weights[:4, 0].detach().numpy().round(3)}")
print(f"    Branch 2: {attn_weights[:4, 1].detach().numpy().round(3)}")

print(f"\nSqueeze-Excitation Fusion:")
print(f"  Output shape: {out_se.shape}")
print(f"  SE weights (first 4 samples):")
print(f"    Branch 1: {se_weights[:4, 0].detach().numpy().round(3)}")
print(f"    Branch 2: {se_weights[:4, 1].detach().numpy().round(3)}")

# 6. Tensor Fusion Network (TFN)

## Concept

The **Tensor Fusion Network** (Zadeh et al., 2017) computes the **outer product** of augmented feature vectors, capturing all unimodal, bimodal, and trimodal interactions in a single tensor.

This is one of the most expressive fusion methods — it explicitly computes all possible multiplicative interactions between features.

## Mathematical Formulation

### Augmented Vectors

First, append a 1 to each feature vector to capture lower-order interactions:

$$\tilde{\mathbf{h}}_1 = [\mathbf{h}_1; 1] \in \mathbb{R}^{d_1 + 1}$$

$$\tilde{\mathbf{h}}_2 = [\mathbf{h}_2; 1] \in \mathbb{R}^{d_2 + 1}$$

### Outer Product

The tensor fusion is the outer product:

$$\mathbf{T} = \tilde{\mathbf{h}}_1 \otimes \tilde{\mathbf{h}}_2 \in \mathbb{R}^{(d_1+1) \times (d_2+1)}$$

Each element:

$$T_{i,j} = \tilde{h}_1^{(i)} \cdot \tilde{h}_2^{(j)}$$

### Decomposition of the Tensor

The resulting tensor contains different types of interactions:

$$\mathbf{T} = \begin{bmatrix} \mathbf{h}_1 \otimes \mathbf{h}_2 & \mathbf{h}_1 \\ \mathbf{h}_2^\top & 1 \end{bmatrix}$$

- $$\mathbf{h}_1 \otimes \mathbf{h}_2$$ — bimodal (cross-stream) interactions
- $$\mathbf{h}_1$$ — unimodal features from branch 1
- $$\mathbf{h}_2$$ — unimodal features from branch 2
- $$1$$ — bias term

The flattened tensor is then projected:

$$\mathbf{y} = \mathbf{W} \cdot \text{vec}(\mathbf{T}) + \mathbf{b}$$

## Complexity

The fused representation has dimensionality $$(d_1+1)(d_2+1)$$, which grows quadratically. For $$d_1 = d_2 = 64$$: the fused vector has $$65 \times 65 = 4225$$ dimensions.

## Low-Rank Tensor Fusion (LMF)

To address the quadratic blowup, **Low-rank Multimodal Fusion** (Liu et al., 2018) decomposes the weight tensor into rank-$$r$$ factors, reducing computation to $$O(r \cdot (d_1 + d_2))$$.

## Industrial Applications

| Industry | Application | Details |
|----------|-------------|---------|
| **Affective Computing** | Multimodal sentiment/emotion analysis | TFN was originally designed for and deployed in sentiment analysis from video (language + audio + visual), used in customer feedback systems (Affectiva, RealEyes) |
| **Human-Computer Interaction** | Emotion-aware dialogue systems | Outer-product fusion of speech prosody, facial expression, and text captures rich cross-modal emotional cues for empathetic chatbots (Soul Machines) |
| **Advertising** | Creative effectiveness prediction | Tensor fusion of visual aesthetics, ad copy semantics, and brand attributes predicts engagement before campaign launch (Nielsen, Kantar) |
| **Security / Surveillance** | Person re-identification | Tensor fusion of RGB appearance, gait patterns, and skeletal pose provides robust identity matching across cameras (Hikvision, Dahua) |
| **Materials Science** | Property prediction | Tensor fusion of crystal structure descriptors and compositional features captures complex structure-property interactions (Citrine Informatics) |

In [0]:
class TensorFusionNetwork(nn.Module):
    """
    Tensor Fusion Network (TFN).
    
    Computes the outer product of augmented vectors to capture all
    unimodal, bimodal, and trimodal interactions.
    
    Reference: Zadeh et al., "Tensor Fusion Network for Multimodal Sentiment Analysis", EMNLP 2017
    """
    def __init__(self, d1: int, d2: int, output_dim: int):
        super().__init__()
        # The fused dimension is (d1+1) * (d2+1) due to augmentation
        fusion_dim = (d1 + 1) * (d2 + 1)
        
        self.output_head = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, output_dim)
        )
    
    def forward(self, h1: torch.Tensor, h2: torch.Tensor) -> torch.Tensor:
        batch_size = h1.size(0)
        
        # Augment with 1s for capturing unimodal terms
        ones = torch.ones(batch_size, 1, device=h1.device)
        h1_aug = torch.cat([h1, ones], dim=-1)  # [B, d1+1]
        h2_aug = torch.cat([h2, ones], dim=-1)  # [B, d2+1]
        
        # Outer product: [B, d1+1, d2+1]
        tensor_product = torch.bmm(
            h1_aug.unsqueeze(2),  # [B, d1+1, 1]
            h2_aug.unsqueeze(1)   # [B, 1, d2+1]
        )
        
        # Flatten: [B, (d1+1)*(d2+1)]
        fused = tensor_product.view(batch_size, -1)
        
        return self.output_head(fused)


class LowRankTensorFusion(nn.Module):
    """
    Low-rank Multimodal Fusion (LMF).
    
    Decomposes the output weight tensor into rank-r factors to avoid
    the quadratic blowup of full tensor fusion.
    
    For each output dimension k:
        y_k = 1^T (W1_k h1_aug ⊙ W2_k h2_aug)
    
    where W1_k ∈ R^{r×(d1+1)}, W2_k ∈ R^{r×(d2+1)}
    
    Reference: Liu et al., "Efficient Low-rank Multimodal Fusion", ACL 2018
    """
    def __init__(self, d1: int, d2: int, output_dim: int, rank: int = 8):
        super().__init__()
        self.rank = rank
        self.output_dim = output_dim
        
        # Low-rank factor matrices
        # Each output dim has its own rank-r decomposition
        self.factor1 = nn.Parameter(torch.randn(output_dim, rank, d1 + 1))
        self.factor2 = nn.Parameter(torch.randn(output_dim, rank, d2 + 1))
        self.bias = nn.Parameter(torch.zeros(output_dim))
        
        # Initialize with Xavier
        nn.init.xavier_normal_(self.factor1)
        nn.init.xavier_normal_(self.factor2)
    
    def forward(self, h1: torch.Tensor, h2: torch.Tensor) -> torch.Tensor:
        batch_size = h1.size(0)
        
        # Augment inputs
        ones = torch.ones(batch_size, 1, device=h1.device)
        h1_aug = torch.cat([h1, ones], dim=-1)  # [B, d1+1]
        h2_aug = torch.cat([h2, ones], dim=-1)  # [B, d2+1]
        
        # Compute factorized tensor fusion
        # factor1: [K, r, d1+1] @ h1_aug: [B, d1+1] -> [K, r, B]
        proj1 = torch.einsum('kri,bi->krb', self.factor1, h1_aug)  # [K, r, B]
        proj2 = torch.einsum('kri,bi->krb', self.factor2, h2_aug)  # [K, r, B]
        
        # Element-wise product and sum over rank: [K, B]
        fusion = (proj1 * proj2).sum(dim=1)  # [K, B]
        
        # Transpose to [B, K] and add bias
        output = fusion.t() + self.bias
        
        return output


# Demonstration
tfn = TensorFusionNetwork(d1=64, d2=64, output_dim=10)
lmf = LowRankTensorFusion(d1=64, d2=64, output_dim=10, rank=8)

out_tfn = tfn(h1, h2)
out_lmf = lmf(h1, h2)

params_tfn = sum(p.numel() for p in tfn.parameters())
params_lmf = sum(p.numel() for p in lmf.parameters())

print(f"Tensor Fusion Network:")
print(f"  Fusion dim:  (64+1)×(64+1) = {65*65}")
print(f"  Output:      {out_tfn.shape}")
print(f"  Parameters:  {params_tfn:,}")

print(f"\nLow-Rank Tensor Fusion (rank=8):")
print(f"  Output:      {out_lmf.shape}")
print(f"  Parameters:  {params_lmf:,}")
print(f"  Reduction:   {params_tfn/params_lmf:.1f}× fewer parameters")

# 7. Cross-Attention Fusion

## Concept

**Cross-Attention** (also called **cross-modal attention**) allows one stream to attend to the other, enabling rich information exchange. Unlike standard self-attention where a sequence attends to itself, cross-attention uses queries from one stream and keys/values from another.

This is the mechanism used in Transformer decoders and is the backbone of models like CLIP, Flamingo, and multimodal LLMs.

## Mathematical Formulation

### Standard Cross-Attention

Given sequences $$\mathbf{H}_1 \in \mathbb{R}^{n_1 \times d}$$ (from stream 1) and $$\mathbf{H}_2 \in \mathbb{R}^{n_2 \times d}$$ (from stream 2):

**Stream 1 attends to Stream 2:**

$$\mathbf{Q} = \mathbf{H}_1 \mathbf{W}_Q, \quad \mathbf{K} = \mathbf{H}_2 \mathbf{W}_K, \quad \mathbf{V} = \mathbf{H}_2 \mathbf{W}_V$$

$$\text{CrossAttn}(\mathbf{H}_1, \mathbf{H}_2) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right)\mathbf{V}$$

where $$\mathbf{W}_Q, \mathbf{W}_K, \mathbf{W}_V \in \mathbb{R}^{d \times d_k}$$.

### Bidirectional Cross-Attention

For symmetric fusion, apply cross-attention in both directions:

$$\hat{\mathbf{H}}_1 = \text{CrossAttn}(\mathbf{H}_1 \to \mathbf{H}_2) \quad \text{(stream 1 queries, stream 2 provides context)}$$

$$\hat{\mathbf{H}}_2 = \text{CrossAttn}(\mathbf{H}_2 \to \mathbf{H}_1) \quad \text{(stream 2 queries, stream 1 provides context)}$$

The fused output can then be obtained by pooling or concatenating $$\hat{\mathbf{H}}_1$$ and $$\hat{\mathbf{H}}_2$$.

### Multi-Head Cross-Attention

Extends to $$h$$ heads for richer representational capacity:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\mathbf{W}_O$$

where each $$\text{head}_i = \text{Attention}(Q\mathbf{W}_Q^i, K\mathbf{W}_K^i, V\mathbf{W}_V^i)$$.

## When to Use Cross-Attention

| Scenario | Example |
|----------|----------|
| Sequence-to-sequence alignment | Machine translation, image captioning |
| Asymmetric modalities | Text querying image regions |
| Fine-grained interaction | Token-level multimodal alignment |
| Variable-length inputs | Different sequence lengths per stream |

## Industrial Applications

| Industry | Application | Details |
|----------|-------------|---------|
| **Large Language Models** | Multimodal LLMs | GPT-4V, Gemini, and LLaVA use cross-attention to allow text tokens to attend to image patch embeddings, enabling visual understanding in production AI assistants |
| **Machine Translation** | Encoder-decoder attention | Google Translate, DeepL use cross-attention as the core mechanism where target language tokens attend to source language representations |
| **Image Captioning / Visual Search** | Region-word alignment | CLIP (OpenAI) and Florence (Microsoft) use cross-attention for fine-grained alignment between image regions and text tokens, powering visual search at scale |
| **Document AI** | Layout-aware understanding | Cross-attention between OCR text tokens and spatial layout features for invoice processing, form extraction (Azure Document Intelligence, AWS Textract) |
| **Drug Discovery** | Molecular interaction modeling | Cross-attention between protein residue sequences and drug molecule tokens for binding site prediction and lead optimization (Recursion, Insilico Medicine) |

In [0]:
class CrossAttentionFusion(nn.Module):
    """
    Bidirectional Cross-Attention Fusion.
    
    Each stream attends to the other using multi-head attention,
    then the enriched representations are pooled and combined.
    
    Used in: CLIP, Flamingo, ViLBERT, LXMERT, Perceiver
    """
    def __init__(self, d_model: int, num_heads: int = 4, output_dim: int = 10):
        super().__init__()
        self.d_model = d_model
        
        # Cross-attention: stream1 attends to stream2
        self.cross_attn_1to2 = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads, batch_first=True
        )
        # Cross-attention: stream2 attends to stream1
        self.cross_attn_2to1 = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads, batch_first=True
        )
        
        # Layer norms (Transformer-style)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Output head
        self.output_head = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.ReLU(),
            nn.Linear(d_model, output_dim)
        )
    
    def forward(self, h1_seq: torch.Tensor, h2_seq: torch.Tensor) -> torch.Tensor:
        """
        Args:
            h1_seq: [B, n1, d_model] - sequence from stream 1
            h2_seq: [B, n2, d_model] - sequence from stream 2
        Returns:
            output: [B, output_dim]
        """
        # Stream 1 attends to Stream 2 (Q=h1, K=V=h2)
        h1_enriched, attn_1to2 = self.cross_attn_1to2(
            query=h1_seq, key=h2_seq, value=h2_seq
        )
        h1_enriched = self.norm1(h1_seq + h1_enriched)  # Residual + norm
        
        # Stream 2 attends to Stream 1 (Q=h2, K=V=h1)
        h2_enriched, attn_2to1 = self.cross_attn_2to1(
            query=h2_seq, key=h1_seq, value=h1_seq
        )
        h2_enriched = self.norm2(h2_seq + h2_enriched)  # Residual + norm
        
        # Pool each stream (mean pooling over sequence length)
        h1_pooled = h1_enriched.mean(dim=1)  # [B, d_model]
        h2_pooled = h2_enriched.mean(dim=1)  # [B, d_model]
        
        # Concatenate pooled representations
        h_fused = torch.cat([h1_pooled, h2_pooled], dim=-1)  # [B, 2*d_model]
        
        return self.output_head(h_fused), (attn_1to2, attn_2to1)


# Demonstration with sequence data
# Simulate: stream 1 has 5 tokens, stream 2 has 7 tokens, both with dim=64
batch_size = 8
d_model = 64

h1_seq = torch.randn(batch_size, 5, d_model)  # e.g., 5 image patches
h2_seq = torch.randn(batch_size, 7, d_model)  # e.g., 7 text tokens

cross_attn_model = CrossAttentionFusion(d_model=64, num_heads=4, output_dim=10)
output, (attn_1to2, attn_2to1) = cross_attn_model(h1_seq, h2_seq)

print(f"Cross-Attention Fusion:")
print(f"  Input stream 1: {h1_seq.shape} (5 tokens)")
print(f"  Input stream 2: {h2_seq.shape} (7 tokens)")
print(f"  Output:          {output.shape}")
print(f"\n  Attention maps:")
print(f"    Stream1→Stream2: {attn_1to2.shape}  (5 queries attend to 7 keys)")
print(f"    Stream2→Stream1: {attn_2to1.shape}  (7 queries attend to 5 keys)")
print(f"\n  Parameters: {sum(p.numel() for p in cross_attn_model.parameters()):,}")

# 8. Feature-wise Linear Modulation (FiLM)

## Concept

**Feature-wise Linear Modulation (FiLM)** (Perez et al., 2018) is an asymmetric fusion mechanism where one stream **modulates** (scales and shifts) the features of the other stream. Rather than treating both streams equally, FiLM designates:

- A **conditioning stream** (generates modulation parameters)
- A **modulated stream** (has its features transformed)

This is ideal when one modality provides **context** that should influence how another modality is processed (e.g., a question modulates how an image is processed in VQA).

## Mathematical Formulation

Given a conditioning vector $$\mathbf{h}_c$$ (e.g., from a text encoder) and features to modulate $$\mathbf{h}_m$$ (e.g., from an image encoder):

**Generate modulation parameters:**

$$\boldsymbol{\gamma} = \mathbf{W}_\gamma \mathbf{h}_c + \mathbf{b}_\gamma \quad \text{(scale)}$$

$$\boldsymbol{\beta} = \mathbf{W}_\beta \mathbf{h}_c + \mathbf{b}_\beta \quad \text{(shift)}$$

**Apply affine transformation:**

$$\mathbf{h}_{\text{fused}} = \boldsymbol{\gamma} \odot \mathbf{h}_m + \boldsymbol{\beta}$$

where $$\boldsymbol{\gamma}, \boldsymbol{\beta} \in \mathbb{R}^d$$.

## Interpretation

- $$\gamma^{(i)} > 1$$: Amplify the $$i$$-th feature
- $$\gamma^{(i)} \approx 0$$: Suppress the $$i$$-th feature
- $$\gamma^{(i)} < 0$$: Invert the $$i$$-th feature
- $$\beta^{(i)}$$: Shift the activation, regardless of the modulated stream's value

## Connection to Other Techniques

| Technique | Relation to FiLM |
|-----------|-------------------|
| Batch Normalization | FiLM with learned $$\gamma, \beta$$ (not input-dependent) |
| Conditional BN | FiLM applied after batch normalization |
| AdaIN (Style Transfer) | FiLM where $$\gamma, \beta$$ come from style statistics |
| Hypernetworks | Generalization where conditioning generates entire weight matrices |

## Industrial Applications

| Industry | Application | Details |
|----------|-------------|---------|
| **Visual Question Answering** | Question-conditioned image processing | The question modulates CNN feature maps so the network "looks at" different image regions depending on what’s asked (Google Lens, Bing Visual Search) |
| **Neural Style Transfer** | Style-conditioned generation | AdaIN (a FiLM variant) lets a style image control the statistics of content features — used in production creative tools (Adobe Photoshop Neural Filters, Prisma) |
| **Text-to-Image Generation** | Prompt-conditioned synthesis | Stable Diffusion and DALL-E use FiLM-like conditioning (adaptive normalization) where text embeddings modulate each layer of the U-Net denoiser |
| **Robotics** | Task-conditioned policies | A task description or goal embedding modulates the robot’s visual processing network so the same backbone handles multiple manipulation tasks (Google DeepMind RT-2) |
| **Game AI** | Context-conditioned behavior | Game state features modulate perception networks so NPCs adapt visual processing to current objectives (Unity ML-Agents, EA SEED) |

In [0]:
class FiLM(nn.Module):
    """
    Feature-wise Linear Modulation (FiLM).
    
    One stream (conditioning) generates scale (γ) and shift (β) parameters
    that modulate the other stream's features.
    
    h_fused = γ ⊙ h_modulated + β
    
    Reference: Perez et al., "FiLM: Visual Reasoning with a General Conditioning Layer", AAAI 2018
    """
    def __init__(self, d_condition: int, d_modulate: int, output_dim: int):
        super().__init__()
        
        # Generate scale and shift from conditioning stream
        self.gamma_net = nn.Linear(d_condition, d_modulate)  # Scale
        self.beta_net = nn.Linear(d_condition, d_modulate)   # Shift
        
        # Output head
        self.output_head = nn.Sequential(
            nn.ReLU(),
            nn.Linear(d_modulate, output_dim)
        )
        
        # Initialize gamma close to 1, beta close to 0
        nn.init.ones_(self.gamma_net.weight.data[:, 0])  
        nn.init.zeros_(self.beta_net.bias.data)
    
    def forward(self, h_condition: torch.Tensor, h_modulate: torch.Tensor) -> torch.Tensor:
        """
        Args:
            h_condition: [B, d_condition] - conditioning stream (generates params)
            h_modulate:  [B, d_modulate] - stream to be modulated
        """
        # Generate modulation parameters from conditioning stream
        gamma = self.gamma_net(h_condition)  # [B, d_modulate] - scale
        beta = self.beta_net(h_condition)    # [B, d_modulate] - shift
        
        # Apply affine modulation
        h_fused = gamma * h_modulate + beta  # [B, d_modulate]
        
        return self.output_head(h_fused), (gamma, beta)


class MultilayerFiLM(nn.Module):
    """
    Multi-layer FiLM: Apply FiLM modulation at multiple layers
    of a processing network (as done in visual reasoning tasks).
    """
    def __init__(self, d_condition: int, d_modulate: int, num_layers: int = 3, output_dim: int = 10):
        super().__init__()
        self.num_layers = num_layers
        
        # FiLM generators for each layer
        self.gamma_nets = nn.ModuleList([nn.Linear(d_condition, d_modulate) for _ in range(num_layers)])
        self.beta_nets = nn.ModuleList([nn.Linear(d_condition, d_modulate) for _ in range(num_layers)])
        
        # Processing layers for the modulated stream
        self.layers = nn.ModuleList([nn.Linear(d_modulate, d_modulate) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_modulate) for _ in range(num_layers)])
        
        self.output_head = nn.Linear(d_modulate, output_dim)
    
    def forward(self, h_condition: torch.Tensor, h_modulate: torch.Tensor) -> torch.Tensor:
        h = h_modulate
        
        for i in range(self.num_layers):
            # Process
            h = self.layers[i](h)
            h = self.norms[i](h)
            
            # FiLM modulation at this layer
            gamma = self.gamma_nets[i](h_condition)
            beta = self.beta_nets[i](h_condition)
            h = gamma * h + beta
            
            # Non-linearity
            h = F.relu(h)
        
        return self.output_head(h)


# Demonstration
# Scenario: h1 is the "conditioning" stream (e.g., question encoding)
#           h2 is the "modulated" stream (e.g., image features)
film = FiLM(d_condition=64, d_modulate=64, output_dim=10)
multi_film = MultilayerFiLM(d_condition=64, d_modulate=64, num_layers=3, output_dim=10)

out_film, (gamma, beta) = film(h_condition=h1, h_modulate=h2)
out_multi = multi_film(h_condition=h1, h_modulate=h2)

print(f"FiLM Fusion:")
print(f"  Output shape: {out_film.shape}")
print(f"  Gamma stats: mean={gamma.mean():.3f}, std={gamma.std():.3f}")
print(f"  Beta stats:  mean={beta.mean():.3f}, std={beta.std():.3f}")
print(f"\nMulti-layer FiLM (3 layers):")
print(f"  Output shape: {out_multi.shape}")
print(f"  Parameters:  {sum(p.numel() for p in multi_film.parameters()):,}")
print(f"\nInterpretation of gamma values:")
print(f"  Dims with |gamma| > 1 (amplified):  {(gamma.abs() > 1).sum().item()}/{gamma.numel()}")
print(f"  Dims with |gamma| < 0.1 (suppressed): {(gamma.abs() < 0.1).sum().item()}/{gamma.numel()}")

# 9. Mixture of Experts (MoE) Fusion

## Concept

**Mixture of Experts (MoE) Fusion** routes the combined input through multiple specialized sub-networks ("experts") and uses a learned **gating network** to weight their outputs. Each expert can specialize in different types of inputs or interaction patterns.

This is especially powerful when the optimal fusion strategy varies across different input regions or data regimes.

## Mathematical Formulation

### Standard MoE

Given $$K$$ expert networks $$\{E_1, E_2, \ldots, E_K\}$$ and a gating network $$G$$:

**Gating weights** (how much to trust each expert for this input):

$$\mathbf{g} = \text{softmax}(\mathbf{W}_g [\mathbf{h}_1 \oplus \mathbf{h}_2] + \mathbf{b}_g) \in \mathbb{R}^K$$

**Expert outputs:**

$$\mathbf{e}_k = E_k([\mathbf{h}_1 \oplus \mathbf{h}_2]) \quad \text{for } k = 1, \ldots, K$$

**Final output** (weighted combination of experts):

$$\mathbf{y} = \sum_{k=1}^K g_k \cdot \mathbf{e}_k$$

### Sparse MoE (Top-k Routing)

For computational efficiency, only activate the top-$$k$$ experts (typically $$k=1$$ or $$k=2$$):

$$\mathbf{g}_{\text{sparse}} = \text{TopK}(\text{softmax}(\mathbf{W}_g \mathbf{x}), k)$$

where $$\text{TopK}$$ zeros out all but the $$k$$ largest entries and renormalizes.

### Load Balancing Loss

To prevent expert collapse (all inputs routed to one expert), add an auxiliary loss:

$$\mathcal{L}_{\text{balance}} = K \cdot \sum_{k=1}^K f_k \cdot p_k$$

where $$f_k$$ is the fraction of inputs routed to expert $$k$$ and $$p_k$$ is the average gate probability for expert $$k$$.

## When to Use MoE Fusion

- **Heterogeneous data**: Different subsets of data benefit from different fusion strategies
- **Large-scale models**: Need capacity without proportional compute increase
- **Multi-task learning**: Different tasks may require different interaction patterns between streams

## Industrial Applications

| Industry | Application | Details |
|----------|-------------|---------|
| **Large-Scale AI** | Foundation model scaling | Google’s Switch Transformer, Mixtral (Mistral AI), and GPT-4 (rumored) use MoE layers to scale model capacity without proportional compute cost |
| **Recommendation Systems** | Multi-domain recommendations | MMoE (Multi-gate MoE) powers YouTube’s recommendation system — different experts specialize in engagement, satisfaction, and diversity objectives simultaneously |
| **Machine Translation** | Language-specialized experts | Different experts activate for different language pairs, enabling a single model to handle 100+ languages efficiently (Google Translate, NLLB by Meta) |
| **Autonomous Driving** | Scenario-specialized perception | Different experts handle highway, urban, and parking scenarios — the gating network routes based on driving context (Wayve, NVIDIA Drive) |
| **Cloud Computing / MLaaS** | Efficient model serving | MoE enables serving very large models cost-effectively since only a fraction of parameters activate per request, reducing GPU memory and latency (Google Cloud Vertex AI, Azure OpenAI) |

In [0]:
class MoEFusion(nn.Module):
    """
    Mixture of Experts Fusion.
    
    Multiple expert networks process the fused input differently,
    and a gating network dynamically weights their outputs.
    
    y = Σ g_k * E_k([h1; h2])
    """
    def __init__(self, d1: int, d2: int, output_dim: int, 
                 num_experts: int = 4, hidden_dim: int = 64, top_k: int = 2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        input_dim = d1 + d2
        
        # Expert networks (each is a small MLP)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, output_dim)
            ) for _ in range(num_experts)
        ])
        
        # Gating network
        self.gate = nn.Sequential(
            nn.Linear(input_dim, num_experts)
        )
        
        # For load balancing loss
        self.register_buffer('expert_counts', torch.zeros(num_experts))
    
    def forward(self, h1: torch.Tensor, h2: torch.Tensor) -> dict:
        """
        Returns dict with 'output', 'gate_weights', and 'load_balance_loss'.
        """
        # Concatenate inputs
        x = torch.cat([h1, h2], dim=-1)  # [B, d1+d2]
        batch_size = x.size(0)
        
        # Compute gate logits and weights
        gate_logits = self.gate(x)  # [B, K]
        
        # Top-k sparse gating
        top_k_logits, top_k_indices = gate_logits.topk(self.top_k, dim=-1)  # [B, top_k]
        top_k_weights = F.softmax(top_k_logits, dim=-1)  # [B, top_k]
        
        # Compute expert outputs (only for selected experts)
        # For simplicity, compute all experts and mask (in practice, use sparse dispatch)
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1)  # [B, K, output_dim]
        
        # Gather top-k expert outputs
        top_k_expert_outputs = torch.gather(
            expert_outputs, 1, 
            top_k_indices.unsqueeze(-1).expand(-1, -1, expert_outputs.size(-1))
        )  # [B, top_k, output_dim]
        
        # Weighted sum of top-k experts
        output = (top_k_weights.unsqueeze(-1) * top_k_expert_outputs).sum(dim=1)  # [B, output_dim]
        
        # Load balancing loss (auxiliary)
        gate_probs = F.softmax(gate_logits, dim=-1)  # [B, K]
        avg_probs = gate_probs.mean(dim=0)  # [K]
        
        # Fraction of inputs where each expert is in top-k
        expert_mask = torch.zeros(batch_size, self.num_experts, device=x.device)
        expert_mask.scatter_(1, top_k_indices, 1.0)
        expert_fractions = expert_mask.mean(dim=0)  # [K]
        
        load_balance_loss = self.num_experts * (avg_probs * expert_fractions).sum()
        
        return {
            'output': output,
            'gate_weights': gate_probs,
            'top_k_indices': top_k_indices,
            'load_balance_loss': load_balance_loss
        }


# Demonstration
moe_fusion = MoEFusion(d1=64, d2=64, output_dim=10, num_experts=4, top_k=2)
result = moe_fusion(h1, h2)

print(f"Mixture of Experts Fusion (K=4, top-k=2):")
print(f"  Output shape: {result['output'].shape}")
print(f"  Load balance loss: {result['load_balance_loss'].item():.4f}")
print(f"  (Ideal balance loss = 1.0; higher means more imbalance)")
print(f"\n  Gate weights (per-expert avg probability):")
avg_gate = result['gate_weights'].mean(dim=0)
for i, w in enumerate(avg_gate):
    print(f"    Expert {i}: {w.item():.3f}")
print(f"\n  Expert routing (which experts handle each sample):")
for i in range(min(4, h1.size(0))):
    experts_used = result['top_k_indices'][i].tolist()
    print(f"    Sample {i}: experts {experts_used}")
print(f"\n  Total parameters: {sum(p.numel() for p in moe_fusion.parameters()):,}")

# 10. Summary and Design Guidelines

## Comprehensive Comparison

| Technique | Parameters | Interaction Order | Symmetric? | Best For |
|-----------|-----------|-------------------|-----------|----------|
| **Late Fusion (Concat)** | $$O(d_{\text{out}}(d_1+d_2))$$ | Implicit (via MLP) | Yes | Default / general purpose |
| **Addition** | 0 | First-order | Yes | Residual connections |
| **Hadamard Product** | 0 | Second-order (aligned) | Yes | Same-space features |
| **Bilinear** | $$O(K \cdot d_1 \cdot d_2)$$ | Second-order (all pairs) | Yes | Small-dim, rich interaction |
| **Low-Rank Bilinear** | $$O(K \cdot r(d_1+d_2))$$ | Approx. second-order | Yes | Large-dim bilinear |
| **Gated** | $$O(d(d_1+d_2))$$ | Dynamic weighting | Yes | Unreliable modalities |
| **Attention** | $$O(d^2)$$ | Soft selection | Yes | Multiple (>2) streams |
| **Tensor Fusion** | $$O(d_{\text{out}}(d_1+1)(d_2+1))$$ | All orders (1st + 2nd) | Yes | Complete interactions |
| **Cross-Attention** | $$O(d^2)$$ per head | Token-level alignment | Bidirectional | Sequences, fine-grained |
| **FiLM** | $$O(d_c \cdot d_m)$$ | Conditional affine | No (asymmetric) | Context modulates features |
| **MoE** | $$O(K \cdot d \cdot d_{\text{out}})$$ | Expert-dependent | Yes | Heterogeneous data |

## Decision Flowchart

1. **Are both streams producing fixed-size vectors?**
   - No (sequences/spatial) → **Cross-Attention** or **FiLM**
   - Yes → Continue

2. **Is one stream clearly "conditioning" the other?**
   - Yes → **FiLM**
   - No → Continue

3. **Do you need explicit feature interactions?**
   - Yes, all pairwise → **Bilinear** or **Tensor Fusion**
   - Yes, aligned only → **Hadamard Product**
   - No → Continue

4. **Does modality reliability vary across samples?**
   - Yes → **Gated Fusion** or **MoE**
   - No → **Late Fusion (Concatenation)** or **Addition**

## Key Takeaways

- **Start simple**: Late fusion (concatenation) is hard to beat for many tasks
- **Match the inductive bias**: Choose a fusion method whose structure matches your domain knowledge
- **Consider the data regime**: With limited data, simpler methods generalize better
- **Combine methods**: Many state-of-the-art models combine multiple fusion strategies at different layers
- **Compute budget matters**: Tensor fusion and full bilinear are expensive; use low-rank approximations

In [0]:
# ============================================================================
# COMPARATIVE BENCHMARK: All Fusion Methods Side by Side
# ============================================================================

import time
from collections import OrderedDict

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def benchmark_forward(model, *inputs, n_iters=100):
    """Measure average forward pass time."""
    # Warmup
    for _ in range(10):
        with torch.no_grad():
            if hasattr(model, 'forward'):
                try:
                    result = model(*inputs)
                except:
                    pass
    # Benchmark
    start = time.time()
    for _ in range(n_iters):
        with torch.no_grad():
            model(*inputs)
    elapsed = (time.time() - start) / n_iters * 1000  # ms
    return elapsed

# Setup: common dimensions
d1, d2, output_dim = 64, 64, 10
batch_size = 32

# Create inputs
torch.manual_seed(42)
x1 = torch.randn(batch_size, d1)
x2 = torch.randn(batch_size, d2)

# Instantiate all models
models = OrderedDict()
models['1. Late Fusion (Concat)'] = LateFusionModel(d1, d2, 128, output_dim)
models['2. Addition'] = ElementWiseFusion(d1, d2, mode='add', output_dim=output_dim)
models['3. Hadamard Product'] = ElementWiseFusion(d1, d2, mode='multiply', output_dim=output_dim)
models['4. Full Bilinear'] = BilinearFusion(d1, d2, output_dim)
models['5. Low-Rank Bilinear (r=16)'] = LowRankBilinearFusion(d1, d2, output_dim, rank=16)
models['6. Gated Fusion'] = GatedFusion(d1, d2, output_dim)
models['7. Tensor Fusion (TFN)'] = TensorFusionNetwork(d1, d2, output_dim)
models['8. FiLM'] = FiLM(d1, d2, output_dim)
models['9. MoE (4 experts, top-2)'] = MoEFusion(d1, d2, output_dim, num_experts=4, top_k=2)

# Benchmark all
print("=" * 80)
print(f"{'FUSION METHOD COMPARISON':^80}")
print(f"{'(batch_size=32, d1=d2=64, output_dim=10)':^80}")
print("=" * 80)
print(f"{'Method':<30} {'Params':>10} {'Time (ms)':>12} {'Output Shape':>15}")
print("-" * 80)

for name, model in models.items():
    model.eval()
    params = count_params(model)
    
    # Forward pass
    with torch.no_grad():
        if 'FiLM' in name:
            output = model(x1, x2)
            if isinstance(output, tuple):
                output = output[0]
        elif 'MoE' in name:
            result = model(x1, x2)
            output = result['output']
        else:
            output = model(x1, x2)
    
    # Benchmark timing
    if 'FiLM' in name:
        t = benchmark_forward(model, x1, x2)
    elif 'MoE' in name:
        t = benchmark_forward(model, x1, x2)
    else:
        t = benchmark_forward(model, x1, x2)
    
    print(f"  {name:<28} {params:>10,} {t:>10.3f}ms {str(list(output.shape)):>15}")

print("=" * 80)
print("\n\u2705 All fusion methods produce the same output shape but differ in:")
print("   • Parameter count (model capacity)")
print("   • Computational cost (inference speed)")
print("   • Inductive bias (what interactions are modeled)")

# 11. Practical Example: End-to-End Multimodal Classification

Below we demonstrate a complete training loop using a **configurable fusion architecture**. This example simulates a multimodal classification task (e.g., combining image and text features to predict sentiment) and shows how different fusion strategies affect convergence.

The architecture follows the standard pattern:

$$\text{Input}_A \xrightarrow{\text{Encoder}_A} \mathbf{h}_1 \xrightarrow{\text{Fusion}} \mathbf{h}_{\text{fused}} \xrightarrow{\text{Classifier}} \hat{y}$$

$$\text{Input}_B \xrightarrow{\text{Encoder}_B} \mathbf{h}_2 \nearrow$$

In [0]:
class MultimodalClassifier(nn.Module):
    """
    Complete multimodal classifier with configurable fusion strategy.
    Demonstrates the standard pattern: Encode -> Fuse -> Classify.
    """
    def __init__(self, input_dim_a: int, input_dim_b: int, 
                 embed_dim: int, num_classes: int, fusion_type: str = "concat"):
        super().__init__()
        self.fusion_type = fusion_type
        
        # Encoders (shared across all fusion types)
        self.encoder_a = nn.Sequential(
            nn.Linear(input_dim_a, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, embed_dim)
        )
        self.encoder_b = nn.Sequential(
            nn.Linear(input_dim_b, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, embed_dim)
        )
        
        # Fusion layer (configurable)
        if fusion_type == "concat":
            self.classifier = nn.Linear(embed_dim * 2, num_classes)
        elif fusion_type == "add":
            self.classifier = nn.Linear(embed_dim, num_classes)
        elif fusion_type == "multiply":
            self.classifier = nn.Linear(embed_dim, num_classes)
        elif fusion_type == "gated":
            self.gate = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.Sigmoid())
            self.classifier = nn.Linear(embed_dim, num_classes)
        elif fusion_type == "bilinear":
            self.bilinear = nn.Bilinear(embed_dim, embed_dim, embed_dim)
            self.classifier = nn.Linear(embed_dim, num_classes)
        elif fusion_type == "film":
            self.gamma_net = nn.Linear(embed_dim, embed_dim)
            self.beta_net = nn.Linear(embed_dim, embed_dim)
            self.classifier = nn.Linear(embed_dim, num_classes)
        else:
            raise ValueError(f"Unknown fusion type: {fusion_type}")
    
    def forward(self, x_a: torch.Tensor, x_b: torch.Tensor) -> torch.Tensor:
        # Encode both modalities
        h1 = self.encoder_a(x_a)  # [B, embed_dim]
        h2 = self.encoder_b(x_b)  # [B, embed_dim]
        
        # Fuse
        if self.fusion_type == "concat":
            h_fused = torch.cat([h1, h2], dim=-1)
        elif self.fusion_type == "add":
            h_fused = h1 + h2
        elif self.fusion_type == "multiply":
            h_fused = h1 * h2
        elif self.fusion_type == "gated":
            g = self.gate(torch.cat([h1, h2], dim=-1))
            h_fused = g * h1 + (1 - g) * h2
        elif self.fusion_type == "bilinear":
            h_fused = self.bilinear(h1, h2)
        elif self.fusion_type == "film":
            gamma = self.gamma_net(h1)
            beta = self.beta_net(h1)
            h_fused = gamma * h2 + beta
        
        # Classify
        return self.classifier(F.relu(h_fused))


# ==========================================================================
# Training experiment: Compare fusion methods on synthetic multimodal data
# ==========================================================================

def create_synthetic_data(n_samples=1000, input_dim_a=50, input_dim_b=30, num_classes=5):
    """Create synthetic data where the label depends on BOTH modalities."""
    torch.manual_seed(42)
    x_a = torch.randn(n_samples, input_dim_a)
    x_b = torch.randn(n_samples, input_dim_b)
    
    # Labels depend on interaction between both modalities
    # (ensures fusion is actually needed)
    combined = torch.cat([x_a[:, :10], x_b[:, :10]], dim=-1)
    interaction = (x_a[:, :5] * x_b[:, :5]).sum(dim=-1)  # multiplicative interaction
    labels = (interaction > interaction.quantile(torch.linspace(0, 1, num_classes + 1)[1:-1].unsqueeze(0)).sum(dim=-1)).long()
    
    return x_a, x_b, labels

def train_model(model, x_a, x_b, labels, epochs=50, lr=0.001):
    """Train and return loss history."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    losses = []
    accs = []
    
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = model(x_a, x_b)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        # Track metrics
        losses.append(loss.item())
        acc = (logits.argmax(dim=-1) == labels).float().mean().item()
        accs.append(acc)
    
    return losses, accs


# Create data
x_a, x_b, labels = create_synthetic_data(n_samples=500, num_classes=5)

# Train with different fusion methods
fusion_types = ["concat", "add", "multiply", "gated", "bilinear", "film"]
results = {}

print("Training multimodal classifiers with different fusion strategies...")
print("=" * 70)

for ft in fusion_types:
    torch.manual_seed(42)  # Same initialization
    model = MultimodalClassifier(
        input_dim_a=50, input_dim_b=30,
        embed_dim=32, num_classes=5, fusion_type=ft
    )
    losses, accs = train_model(model, x_a, x_b, labels, epochs=100)
    results[ft] = {'losses': losses, 'accs': accs, 'params': count_params(model)}
    print(f"  {ft:<12} | Final loss: {losses[-1]:.4f} | Final acc: {accs[-1]:.3f} | Params: {count_params(model):,}")

print("=" * 70)
print("\n\u2705 Training complete! Results show how fusion choice affects learning.")
print("   Note: 'multiply' and 'bilinear' often perform better when labels")
print("   depend on multiplicative interactions between modalities (as here).")

# References and Further Reading

## Key Papers

1. **Late Fusion / Two-Stream Networks**: Simonyan & Zisserman, *Two-Stream Convolutional Networks for Action Recognition in Videos*, NeurIPS 2014

2. **Bilinear Pooling**: Lin et al., *Bilinear CNN Models for Fine-grained Visual Recognition*, ICCV 2015

3. **Low-Rank Bilinear (MCB/MLB)**: 
   - Kim et al., *Hadamard Product for Low-rank Bilinear Pooling*, ICLR 2017
   - Fukui et al., *Multimodal Compact Bilinear Pooling*, EMNLP 2016

4. **Tensor Fusion Network**: Zadeh et al., *Tensor Fusion Network for Multimodal Sentiment Analysis*, EMNLP 2017

5. **Low-Rank Multimodal Fusion**: Liu et al., *Efficient Low-rank Multimodal Fusion with Modality-Specific Factors*, ACL 2018

6. **FiLM**: Perez et al., *FiLM: Visual Reasoning with a General Conditioning Layer*, AAAI 2018

7. **Cross-Attention**:
   - Vaswani et al., *Attention Is All You Need*, NeurIPS 2017
   - Lu et al., *ViLBERT: Pretraining Task-Agnostic Visiolinguistic Representations*, NeurIPS 2019

8. **Gated Multimodal Units**: Arevalo et al., *Gated Multimodal Units for Information Fusion*, ICLR Workshop 2017

9. **Mixture of Experts**: 
   - Shazeer et al., *Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer*, ICLR 2017
   - Fedus et al., *Switch Transformers: Scaling to Trillion Parameter Models*, JMLR 2022

10. **Squeeze-and-Excitation**: Hu et al., *Squeeze-and-Excitation Networks*, CVPR 2018

## Survey Papers

- Baltrušaitis et al., *Multimodal Machine Learning: A Survey and Taxonomy*, TPAMI 2019
- Gao et al., *A Survey on Deep Multimodal Learning for Body Language Recognition and Generation*, 2022